# Logistic Regression Baseline Model

This notebook builds a baseline machine learning model for the crypto risk project.

The goal is to classify each Bitcoin market moment as:

- Low risk
- Medium risk
- High risk

This model is not the final neural network. It is a baseline model used for comparison later.

In [ ]:
# Import necessary files
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 1. Load the Labeled Dataset

This section loads the dataset that already contains:

- cleaned Bitcoin price data
- engineered features
- risk labels

The model will learn patterns from these features to predict the risk label.

In [ ]:
# Read labeled file
df = pd.read_csv("../Data_Files/btc_1h_labeled.csv")
# Convert timestamp to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])
# Sort by timestamp and reset index
df = df.sort_values("timestamp").reset_index(drop=True)
# Display the first few rows of the DataFrame
df.head()

,timestamp,open,high,low,close,volume,return_1h,return_3h,return_6h,return_12h,...,macd_histogram,sma_20_50_ratio,trend_strength,volatility_ratio_24h_72h,true_range,atr_14,atr_pct,downside_volatility_24h,future_drawdown_48h,buy_signal
0,2022-01-07 23:00:00+00:00,41498.42,41586.03,41333.33,41557.73,28.204079,0.000823,-0.008587,-0.010164,-0.016692,...,29.228447,0.983239,-0.016761,1.264688,252.70,597.740714,0.014383,0.006908,-0.025033,1
1,2022-01-08 00:00:00+00:00,41557.73,42107.20,41557.73,41971.07,29.026462,0.009946,0.002365,0.006624,-0.010564,...,52.027664,0.984259,-0.015741,1.288050,549.47,612.157857,0.014585,0.006944,-0.034635,0
2,2022-01-08 01:00:00+00:00,41987.47,42231.97,41857.23,41918.98,24.323575,-0.001241,0.009523,0.003726,0.010726,...,63.285497,0.985071,-0.014929,1.278487,374.74,626.020000,0.014934,0.006916,-0.033435,0
3,2022-01-08 02:00:00+00:00,41896.06,41944.67,41704.00,41876.14,21.664747,-0.001022,0.007662,-0.000991,-0.000418,...,67.165334,0.986314,-0.013686,1.277565,240.67,627.632857,0.014988,0.006947,-0.032447,0
4,2022-01-08 03:00:00+00:00,41878.69,41908.18,41666.32,41797.90,20.451643,-0.001868,-0.004126,-0.001771,0.011431,...,63.805170,0.987243,-0.012757,1.106302,241.86,526.890000,0.012606,0.005403,-0.030636,0


## 2. Inspect the Dataset

Before training the model, we check the size of the dataset and look at the available columns.

In [ ]:
# Print dataset information
print("Dataset shape:")
print(df.shape)
# Print column names
print("\nColumns:")
print(df.columns.tolist())

Dataset shape:
(38050, 39)

Columns:
['timestamp', 'open', 'high', 'low', 'close', 'volume', 'return_1h', 'return_3h', 'return_6h', 'return_12h', 'return_24h', 'volatility_6h', 'volatility_24h', 'volatility_72h', 'sma_20', 'sma_50', 'distance_from_sma_20', 'distance_from_sma_50', 'volume_change_1h', 'volume_change_24h', 'volume_sma_24', 'volume_ratio_24h', 'rolling_high_24h', 'rolling_high_7d', 'drawdown_from_24h_high', 'drawdown_from_7d_high', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'sma_20_50_ratio', 'trend_strength', 'volatility_ratio_24h_72h', 'true_range', 'atr_14', 'atr_pct', 'downside_volatility_24h', 'future_drawdown_48h', 'buy_signal']


## 3. Check Class Balance

This shows how many examples belong to each risk category.

Class balance is important because financial datasets often contain many more low-risk moments than high-risk moments.

In [ ]:
print("Buy signal counts:")
print(df["buy_signal"].value_counts())

print("\nBuy signal percentages:")
print(df["buy_signal"].value_counts(normalize=True))

Buy signal counts:
buy_signal
1    25359
0    12691
Name: count, dtype: int64

Buy signal percentages:
buy_signal
1    0.666465
0    0.333535
Name: proportion, dtype: float64


## 4. Encode Risk Labels

Machine learning models need numeric labels.

This converts:

- High
- Low
- Medium

into numbers.

In [ ]:
print("Buy Signal Mapping:")
print("1 = Low-Risk Buy Opportunity")
print("0 = Not Low-Risk")

Buy Signal Mapping:
1 = Low-Risk Buy Opportunity
0 = Not Low-Risk


## 5. Select Feature Columns

These are the input variables the model uses to predict risk.

They include returns, volatility, RSI, MACD, volume changes, moving average distance, and drawdown features.

In [ ]:
feature_columns = [
    "trend_strength",
    "volatility_ratio_24h_72h",
    "atr_pct",
    "downside_volatility_24h",
    "return_1h",
    "return_3h",
    "return_6h",
    "return_12h",
    "return_24h",
    "volatility_6h",
    "volatility_24h",
    "volatility_72h",
    "rsi_14",
    "macd",
    "macd_signal",
    "macd_histogram",
    "volume_change_1h",
    "volume_change_24h",
    "volume_ratio_24h",
    "distance_from_sma_20",
    "distance_from_sma_50",
    "drawdown_from_24h_high",
    "drawdown_from_7d_high",
]

## 6. Remove Missing Values

Some technical indicators create missing values at the beginning of the dataset.

Rows with missing feature values are removed before training.

In [215]:
print("Missing values before cleaning:")
print(df[feature_columns + ["risk_label"]].isna().sum())

df = df.dropna(subset=feature_columns + ["buy_signal"]).reset_index(drop=True)

print("\nDataset shape after removing missing values:")
print(df.shape)

Missing values before cleaning:


KeyError: "['risk_label'] not in index"

## 7. Create X and y

X contains the input features.

y contains the target risk label the model is trying to predict.

In [ ]:
X = df[feature_columns]
y = df["buy_signal"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (38050, 23)
y shape: (38050,)


## 8. Chronological Train/Test Split

Because this is time-series market data, the dataset is split chronologically.

The first 80% is used for training.

The last 20% is used for testing.

This prevents future data from leaking into the training set.

In [ ]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 30440
Testing rows: 7610


## 9. Scale the Features

Logistic regression performs better when features are on a similar scale.

The scaler is fit only on the training data.

The same scaler is then applied to the test data.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 10. Train the Logistic Regression Model

This trains the baseline classification model.

`class_weight="balanced"` helps the model pay more attention to rare classes, especially High-risk examples.


In [ ]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

## 11. Make Predictions

The trained model predicts the risk label for the test set.

In [ ]:
y_pred = model.predict(X_test_scaled)

## 12. Evaluate the Model

This section measures model performance.

Accuracy shows overall correctness.

The confusion matrix shows where the model gets confused.

The classification report shows precision, recall, and F1-score for each risk class.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:")
print(accuracy)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Low-Risk", "Low-Risk"]
    )
)

Accuracy:
0.5701708278580815

Confusion Matrix:
[[ 142  105   38]
 [ 815 3859  631]
 [ 536 1146  338]]

Classification Report:
              precision    recall  f1-score   support

        High       0.10      0.50      0.16       285
         Low       0.76      0.73      0.74      5305
      Medium       0.34      0.17      0.22      2020

    accuracy                           0.57      7610
   macro avg       0.40      0.46      0.37      7610
weighted avg       0.62      0.57      0.58      7610



## 13. Prediction Probabilities

Instead of only predicting one class, the model can output probabilities for each risk class.

This is useful later for showing confidence in the dashboard.

In [ ]:
probabilities = model.predict_proba(X_test_scaled)

df_predictions = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred,
    "Low_Risk_Probability": probabilities[:, 1]
})

df_predictions.head()

Example Prediction Probabilities:

Example 1:
High: 0.1319
Low: 0.5518
Medium: 0.3163

Example 2:
High: 0.1659
Low: 0.5339
Medium: 0.3002

Example 3:
High: 0.1874
Low: 0.5499
Medium: 0.2627

Example 4:
High: 0.2059
Low: 0.5193
Medium: 0.2747

Example 5:
High: 0.2118
Low: 0.5177
Medium: 0.2706


## 14. Analyze Prediction Confidence

The model does not only predict a risk class. It also produces probabilities for each class.

These probabilities can be used to measure confidence in a prediction.

For real-world investing, confidence is important because a trader may only want to enter a position when the model is highly confident that risk is low.

In [ ]:
print(df_predictions["Low_Risk_Probability"].describe())

print("\nMaximum Low-Risk Probability:")
print(df_predictions["Low_Risk_Probability"].max())

,Actual,Predicted,Low_Probability
0,Low,Low,0.551826
1,Low,Low,0.533884
2,Low,Low,0.549884
3,Low,Low,0.519339
4,Low,Low,0.517651


## 15. Evaluate High-Confidence Low-Risk Predictions

The most important use case for this project is identifying situations where buying a cryptocurrency appears relatively safe.

This section isolates predictions where:

- The model predicts "Low" risk
- The model is highly confident in that prediction

By examining only these predictions, we can estimate how trustworthy a high-confidence Low-risk signal may be in a real trading environment.

In [ ]:
threshold = 0.70

high_confidence_low = df_predictions[
    df_predictions["Low_Risk_Probability"] >= threshold
]

print("Threshold:", threshold)
print("High-confidence low-risk predictions:", len(high_confidence_low))

high_confidence_low.head()

High Confidence Low-Risk Predictions:
6
count    7610.000000
mean        0.387601
std         0.094778
min         0.020563
25%         0.329753
50%         0.399337
75%         0.457643
max         0.675465
Name: Low_Probability, dtype: float64
0.6754648720982042


## 16. Measure High-Confidence Low-Risk Accuracy

This metric answers the question:

"When the model is highly confident that risk is low, how often is it correct?"

This is one of the most practically useful metrics for a risk-management system because it evaluates the reliability of potential buy signals rather than overall model accuracy.

In [ ]:
if len(high_confidence_low) > 0:
    high_confidence_accuracy = (
        high_confidence_low["Actual"] == 1
    ).mean()

    print(
        f"High-Confidence Low-Risk Accuracy: "
        f"{high_confidence_accuracy:.2%}"
    )
else:
    print("No high-confidence low-risk predictions at this threshold.")

High Confidence Low-Risk Accuracy: nan%


# 17. Distribution of Low-Risk Confidence Scores

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.hist(
    df_predictions["Low_Risk_Probability"],
    bins=30
)

plt.title("Distribution of Low-Risk Prediction Probabilities")
plt.xlabel("Probability of Low-Risk Buy Signal")
plt.ylabel("Count")

plt.show()

## 18. Test Multiple Confidence Thresholds

This section evaluates how reliable the model is at different low-risk probability thresholds.

A higher threshold creates fewer buy signals, but those signals should ideally be more accurate.

In [ ]:
for threshold in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
    signals = df_predictions[
        df_predictions["Low_Risk_Probability"] >= threshold
    ]

    if len(signals) == 0:
        print(f"Threshold {threshold}: 0 signals")
        continue

    signal_accuracy = (signals["Actual"] == 1).mean()

    print(
        f"Threshold {threshold}: "
        f"{len(signals)} signals, "
        f"{signal_accuracy:.2%} actually low-risk"
    )

## 19. Feature Importance

Logistic regression coefficients show how strongly each feature influences each class.

Positive coefficients push the prediction toward that class.

Negative coefficients push the prediction away from that class.

In [ ]:
importance = pd.DataFrame({
    "Feature": feature_columns,
    "Coefficient": model.coef_[0]
})

importance = importance.sort_values(
    by="Coefficient",
    ascending=False
)

display(importance)


===== High =====


,Feature,Coefficient
20,distance_from_sma_50,2.137561
8,return_24h,0.163910
10,volatility_24h,0.085439
9,volatility_6h,0.076180
3,downside_volatility_24h,0.057363
16,volume_change_1h,0.043116
14,macd_signal,0.041891
13,macd,0.038665
7,return_12h,0.022250
2,atr_pct,0.017965



===== Low =====


,Feature,Coefficient
0,trend_strength,0.818742
19,distance_from_sma_20,0.531434
17,volume_change_24h,0.186258
22,drawdown_from_7d_high,0.155879
12,rsi_14,0.144800
21,drawdown_from_24h_high,0.114020
3,downside_volatility_24h,0.063103
6,return_6h,0.052547
1,volatility_ratio_24h_72h,0.047424
16,volume_change_1h,0.036345



===== Medium =====


,Feature,Coefficient
0,trend_strength,0.583750
19,distance_from_sma_20,0.527091
11,volatility_72h,0.280286
17,volume_change_24h,0.176574
21,drawdown_from_24h_high,0.126880
22,drawdown_from_7d_high,0.118829
2,atr_pct,0.115185
1,volatility_ratio_24h_72h,0.089189
12,rsi_14,0.068851
18,volume_ratio_24h,0.055572


## 20. Save the Model

The trained model, scaler, and label encoder are saved so they can be reused later.

These files will be needed for the Streamlit dashboard.

In [ ]:
os.makedirs("../Models", exist_ok=True)

joblib.dump(model, "../Models/low_risk_logistic_model.pkl")
joblib.dump(scaler, "../Models/low_risk_scaler.pkl")

print("Low-risk logistic regression model and scaler saved successfully.")

Model files saved successfully.


# Logistic Regression Baseline Model Summary

This notebook developed and evaluated a logistic regression baseline model for cryptocurrency risk classification. The objective was to predict whether a market condition should be classified as Low, Medium, or High risk based on engineered technical indicators and future drawdown behavior.

The model was trained using features derived from historical Bitcoin price and volume data, including returns, volatility metrics, RSI, MACD, moving-average relationships, volume analysis, drawdown measurements, ATR, trend strength, and downside volatility.

## Key Results

- Achieved approximately 57% classification accuracy using the original risk-labeling methodology.
- Successfully identified meaningful relationships between market indicators and future downside risk.
- Demonstrated that feature engineering improved model performance, although gains became incremental as additional indicators were added.
- Revealed that model performance was highly sensitive to risk-label definitions, highlighting the importance of target design in financial machine learning.
- Produced probabilistic risk estimates that can be used to evaluate prediction confidence rather than relying solely on class labels.

## Lessons Learned

Several labeling strategies were tested, including fixed drawdown thresholds and quantile-based risk labels. These experiments showed that modifying the target definition often had a larger impact on model behavior than adding additional technical indicators.

The model also demonstrated relatively low confidence in many predictions, suggesting that cryptocurrency risk classification is a complex problem with overlapping market conditions and nonlinear relationships.

## Project Outcome

The logistic regression model serves as a baseline benchmark for the project. While it was able to capture meaningful market patterns, its limitations indicate that more flexible models may better represent the nonlinear behavior present in cryptocurrency markets.

The next phase of the project is to build and evaluate a neural network using the same dataset and features. Performance will be compared against the logistic regression baseline to determine whether a neural network can improve classification accuracy, confidence, and overall risk assessment capability.